In [ ]:
#imports here 
import pandas as pd
import numpy as np
import os
import cv2
import pydicom
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
import glob
from tqdm import tqdm
import math
import pydicom as dicom

In [ ]:
#EDA on 1st dataset(worldwidecovid)
path_data_1_combined = "/kaggle/input/world-wide-covid-dataset/PROSTATE_MRI"
#read metadata sheet
df = pd.read_csv(os.path.join(path_data_1_combined,'metadata.csv'))

In [ ]:
print(df.head())
#this tells us that multiple mri scans have been taken for each patient

In [ ]:
print(df)

In [ ]:
#average number of scans per patient 
#for that pick the number of columns 
#this sheet can be used for us to iterate the files as necessary
#SubjectID identifies the patient
print("Number of Rows:",len(df["Number of Images"]))
#indicate the number of unique patients

In [ ]:
total = 0
for rowdata in df['Number of Images']:
    total+=rowdata
print("The Average number of MRI scans each patient underwent are:",total/182)

In [ ]:
#methodologies used and ranking
methods_unique = set()
for method in df['Series Description']:
    methods_unique.add(method)

In [ ]:
print("Protocols used:",methods_unique)

In [ ]:
print("Number of unique protocols used:",len(methods_unique))

In [ ]:
#find out the most used technique for mri
max_count = 0
count = 0
method_max = ''
for ele in methods_unique:
    for method in df['Series Description']:
        if(ele == method):
            count+=1
    if count>=max_count:
        max_count = count
        count = 0
        method_max = ele
        print(method_max)
        print(max_count)
print("The most used method is:",method_max)

In [ ]:
# Path to the combined directory containing DICOM files
combined_path = os.path.join(path_data_1_combined, "PROSTATE-MRI")

# Iterate over each directory in the combined path
for direct in os.listdir(combined_path):
    image_path = os.path.join(combined_path, direct)
    
    # Only process directories (you might want to add additional checks here)
    if os.path.isdir(image_path) and direct != 'LICENSE':
        counter = 0
        # Use os.walk to iterate through the files in the directory
        for dirpath, dirnames, filenames in os.walk(image_path):
            # Only read one DICOM file
            for filename in filenames:
                # Construct the full path to the DICOM file
                file_path = os.path.join(dirpath, filename)
                
                # Check if the file is a DICOM file
                if filename.lower().endswith('.dcm') and counter < 1:
                    ds = pydicom.dcmread(file_path)
                    # Access pixel data
                    if 'PixelData' in ds:
                        pixel_array = ds.pixel_array

                        # Display the image
                        plt.imshow(pixel_array, cmap='gray')  # Use 'gray' colormap for grayscale images
                        plt.title('DICOM Image')
                        plt.axis('off')  # Hide axes
                        plt.show()
                        counter += 1
                    else:
                        print("No pixel data found in this DICOM file.")


In [ ]:
#eda on prostatemriusbiopsy
path_to_2_combined = "/kaggle/input/prostate-mri-us-biopsy"
df2 = pd.read_csv(os.path.join(path_to_2_combined , 'metadata.csv'))

In [ ]:
print(df2.head())

In [ ]:
excel_for_dataset = '/kaggle/input/prostate-mri-us-biopsy/TCIA Biopsy Data_2020-07-14.xlsx'
df3 = pd.read_excel(excel_for_dataset)

In [ ]:
print(df3.head())

In [ ]:
print(df3)

In [ ]:
print(df3.columns)

In [ ]:
excel_dataset_2 = "/kaggle/input/prostate-mri-us-biopsy/Target Data_2019-12-05.xlsx"
df4 = pd.read_excel(excel_dataset_2)

In [ ]:
print(df4.head())

In [ ]:
print("Number of Rows:",len(df2["Number of Images"]))

In [ ]:
total = 0
for rowdata in df2['Number of Images']:
    total+=rowdata
print("The Average number of MRI scans each patient underwent are:",total/2779)

In [ ]:
#methodologies used and ranking
methods_unique2 = set()
for method in df2['Series Description']:
    methods_unique2.add(method)

In [ ]:
print("Protocols used:",methods_unique2)

In [ ]:
print("Number of unique protocols used:",len(methods_unique2))

In [ ]:
#find out the most used technique for mri
max_count = 0
count = 0
method_max = ''
for ele in methods_unique2:
    for method in df2['Series Description']:
        if(ele == method):
            count+=1
    if count>=max_count:
        max_count = count
        count = 0
        method_max = ele
        print(method_max)
        print(max_count)
print("The most used method is:",method_max)

In [ ]:
path_to_stl = "/kaggle/input/prostate-mri-us-biopsy/STLs/STLs/"
counter_stl = 0

for stlfile in os.listdir(path_to_stl):
    counter_stl+=1
print("The Number of stl ultrasound files are:",counter_stl)

In [ ]:
# Path to the MRI biopsy directory
path_to_mri_biopsy = '/kaggle/input/prostate-mri-us-biopsy/prostate_mri_us_biopsy/Prostate-MRI-US-Biopsy/'
limit = 25
counter_limit = 0

# Iterate through each directory in the main directory
for directory in os.listdir(path_to_mri_biopsy):
    directory_path = os.path.join(path_to_mri_biopsy, directory)
    
    # Check if it's a directory
    if os.path.isdir(directory_path) and counter_limit < limit:
        # Use os.walk to traverse subdirectories
        for dirpath, dirnames, filenames in os.walk(directory_path):
            # Find the first DICOM file
            for filename in filenames:
                # Construct the full path to the DICOM file
                if filename.lower().endswith('.dcm'):  # Check for DICOM files
                    file_path = os.path.join(dirpath, filename)
                    print(f"Reading DICOM file: {file_path}")
                    
                    ds = pydicom.dcmread(file_path)
                    
                    # Access pixel data
                    if 'PixelData' in ds:
                        pixel_array = ds.pixel_array

                        # Check the shape of the pixel array
                        if pixel_array.ndim == 3:
                            # If it's 3D, display the middle slice
                            mid_slice = pixel_array[pixel_array.shape[0] // 2, :, :]
                            plt.imshow(mid_slice, cmap='gray')
                        elif pixel_array.ndim == 2:
                            # If it's 2D, display it directly
                            plt.imshow(pixel_array, cmap='gray')
                        else:
                            print("Unexpected number of dimensions in pixel data.")
                            
                        plt.title(f'DICOM Image from {directory}')
                        plt.axis('off')  # Hide axes
                        plt.show()
                        
                        counter_limit += 1  # Increment the counter
                        break  # Break after showing the first DICOM image for this directory

                if counter_limit >= limit:  # Check if limit is reached
                    break  # Exit the loop if limit is reached

    if counter_limit >= limit:  # Check if limit is reached
        break  # Exit the outer loop if limit is reached

In [ ]:
# Merge DataFrames on different column names
combined_df = pd.merge(
    df3, df4,
    left_on=['Patient Number', 'Series Instance UID (US)', 'Series Instance UID (MRI)'],
    right_on=['Patient ID', 'seriesInstanceUID_US', 'seriesInstanceUID_MR'],
    how='inner'
)

# Drop unnecessary columns after merging
combined_df.drop(['Patient Number' ,
                  'Series Instance UID (US)', 'seriesInstanceUID_US', 
                  'Series Instance UID (MRI)', 'seriesInstanceUID_MR'], 
                 axis=1, inplace=True, errors='ignore')

# Print the combined DataFrame
print(combined_df)


In [ ]:
print(combined_df.columns)

In [ ]:
print(combined_df.head())

In [ ]:
print(df3.shape)

In [ ]:
print(df4.shape)

In [ ]:
print(combined_df.columns.tolist())

In [ ]:
combined_df.columns = combined_df.columns.str.strip()

In [ ]:
print(combined_df.columns.tolist())

In [ ]:
#label encode the columns that are categorical 
encoder = LabelEncoder()

combined_df['Core Label Encoded'] = encoder.fit_transform(combined_df['Core Label'])

combined_df.drop('Core Label', axis=1, inplace=True)

print(combined_df)

In [ ]:
columns_to_fill = ['Secondary Gleason','Cancer Length (mm)','% Cancer in Core', 'Core Fragment #1 Tissue Length (mm)', 'Core Fragment #2 Tissue Length (mm)' , 'Core Fragment #3 Tissue Length (mm)']

for column in columns_to_fill:
    combined_df[column] = combined_df[column].fillna(0)

In [ ]:
combined_df.head()

In [ ]:
combined_df['Primary Gleason'].fillna(combined_df['Primary Gleason'].mode()[0], inplace=True)

In [ ]:
print(combined_df)

In [ ]:
# Calculate the correlation matrix
corr = combined_df.corr()

# Select correlations with UCLA Score
ucla_corr = corr[['UCLA Score (Similar to PIRADS v2)']]

# Create the heatmap
plt.figure(figsize=(10, 8))  # Adjust the figure size as needed
sns.heatmap(ucla_corr, annot=True, cmap='coolwarm', linewidths=0.5)

# Save the plot
plt.savefig('ucla_score_corr_map.jpg', dpi=300)  # Save the heatmap as a JPEG file

# Show the plot
plt.show()  # Display the heatmap

In [ ]:
# Config Create Images
images_to_use = 25
x_crop = 56
y_crop = 56
folder_path = "/kaggle/input/prostate-mri-us-biopsy/prostate_mri_us_biopsy/Prostate-MRI-US-Biopsy"
output_folder_path = "/kaggle/working"

In [ ]:
def process_folder(folder_path):
    
    """
    process_folder processes DCOM images in the folder
    
    :param folder_path: folder to process
    :return: jpg_list,dcm_values
    """ 
    
    jpg_list = []
    dcm_values = []
    
    images_path = os.listdir(folder_path)

    for image in images_path:
        full_path = os.path.join(folder_path, image)
        if image.endswith(".dcm"):
            ds = dicom.dcmread(full_path)
        
            jpg_image_name = output_folder_path+full_path+".jpg"
            jpg_list.append(jpg_image_name)
                       
            if not os.path.exists(output_folder_path+folder_path):
                os.makedirs(output_folder_path+folder_path)
            
            cv2.imwrite(jpg_image_name, ds.pixel_array)
                               
            patient_name=''
            patient_age=''
            patient_size=''
            patient_weight=''
            patient_eth=''
            patient_occ=''
            patient_smoke=''
            
            try:
                patient_name = ds[0x0010,0x0010].value
            except:
                pass
            
            try:
                patient_age = ds[0x0010,0x1010].value
            except:
                pass
            
            try:
                patient_size = ds[0x0010,0x1020].value
            except:
                pass

            try:
                patient_weight = ds[0x0010,0x1030].value
            except:
                pass
            
            try:
                patient_eth = ds[0x0010,0x2160].value
            except:
                pass            
            
            try:
                patient_occ = ds[0x0010,0x2180].value
            except:
                pass            
            
            try:
                patient_smoke = ds[0x0010,0x21a0].value
            except:
                pass  
            
            dcm_values.append([patient_name,patient_age,patient_size,patient_weight,patient_eth,patient_occ,patient_smoke])
            
    return jpg_list,dcm_values,(output_folder_path+folder_path)

In [ ]:
def crop_image(folder_path,x_crop,y_crop):
    
    """
    crop_image crops the image
    
    :param folder_path: folder to process
    :param x_crop: pixel crop x
    :param y_crop: pixel crop y
    """ 
    
    fileList = glob.glob(folder_path + '*jpg*')
    
    # Iterate over images
    for filePath in fileList:
        img = cv2.imread(filePath)
        image_shape_x = img.shape[0]
        image_shape_y = img.shape[1]
                        
        crop_img = img[(0+y_crop):(image_shape_y-y_crop), (0+x_crop):(image_shape_x-x_crop)]
        cv2.imwrite(filePath, crop_img)

In [ ]:
def remove_jpg(folder_path):
  
    """
    remove_jpg removes jpg from folder
    
    :param folder_path: folder to process
    """ 
    
    # ==========================================================================================
    # Remove old Final Files
    # ==========================================================================================
    fileList1  = glob.glob(folder_path + '**/*AX_T2*/*jpg*' , recursive=True)
    fileList2 = glob.glob(folder_path + '**/*PROPELLER*/*jpg*' , recursive=True)
    fileList3 = glob.glob(folder_path + '**/*axial*/*jpg*' , recursive=True)
    fileList4 = glob.glob(folder_path + '**/*T2 Ax*/*jpg*' , recursive=True)
    fileList5 = glob.glob(folder_path + '**/*AX T2*/*jpg*' , recursive=True)
    fileList6 = glob.glob(folder_path + '**/*AX */*jpg*' , recursive=True)
    fileList7 = glob.glob(folder_path + '**/*T2 AX*/*jpg*' , recursive=True)
    fileList8 = glob.glob(folder_path + '**/*T2 SPACE*/*jpg*' , recursive=True)
    fileList = fileList1 + fileList2 + fileList3 + fileList4 + fileList5 + fileList6 + fileList7 + fileList8  
        
    # Iterate over the list of filepaths & remove each file.
    for filePath in fileList:    
        try:            
            os.remove(filePath)
        except:
            pass

In [ ]:
def make_patient_image(folder_path,images_to_use):
    
    """
    make_patient_image creates images
    
    :param folder_path: folder to process
    :param images_to_use: images to use
    """ 
               
    # ==========================================================================================
    # List all Images
    # ==========================================================================================
    images_path = os.listdir(folder_path)
    image_list = []
        
    for image in images_path:
        full_path = os.path.join(folder_path, image)
        if image.endswith(".jpg"):
            image_list.append(full_path)            
    
    #sort images
    image_list.sort()
    
    image_cnt = len(image_list)
    
    image_ignore = math.floor((image_cnt- images_to_use)/2)
            
    #remove first x images
    image_list = image_list[image_ignore:]
    
    #remove last x images
    image_list = image_list[:len(image_list)-image_ignore]
    
    image_list = image_list[:images_to_use]
            
    image_cnt = len(image_list)
    
    if image_cnt == 25:
        dims = 5
    elif image_cnt == 16:
        dims = 4
    elif image_cnt == 4:
        dims = 2
    elif image_cnt < 4:
        dims = 1        
    else:
        dims = math.ceil((math.sqrt(image_cnt)))   
                              
    # ==========================================================================================
    # Looping Over Images to Concatenate and create Vertical
    # ==========================================================================================
    for coll_cnt in range(0,dims):

        final_image = os.path.join(folder_path, "finalV" + str(coll_cnt)+".jpg")
        zero_flag = 0
        
        #print("======================================")
        
        for x in range(1,(dims)):        

            image_number = (x - 1) + (coll_cnt * dims)

            if image_number >= (image_cnt-1):
                # set image file names
                img1 = final_image
                img2 = image_list[(0)]

            elif zero_flag == 0:
                # set image file names
                img1 = image_list[image_number]
                img2 = image_list[(image_number+1)]            
                zero_flag = 1
            else:
                # set image file names
                img1 = final_image
                img2 = image_list[(image_number+1)]
                
            # read the images           
            img1 = cv2.imread(img1)
            img2 = cv2.imread(img2)
            
            try:
                im_v = cv2.vconcat([img1, img2])   
                cv2.imwrite(final_image, im_v)        
            except:
                pass
                        
    # ==========================================================================================
    # Looping Over finalV images to concatenate into final image
    # ==========================================================================================    
    
    final_image = os.path.join(folder_path, "final.jpg")
    image_list = glob.glob(folder_path + '*final*')
    image_cnt = len(image_list) - 1
    zero_flag = 0
    image_list.sort()
        
    for x in range(0,image_cnt):

        if zero_flag == 0:
            # set image file names
            img1 = image_list[x]
            img2 = image_list[(x+1)]            
            zero_flag = 1
        else:
            # set image file names
            img1 = final_image
            img2 = image_list[(x+1)]
            
        # read the images
        img1 = cv2.imread(img1)
        img2 = cv2.imread(img2)

        im_v = cv2.hconcat([img1, img2])   
        cv2.imwrite(final_image, im_v)

In [ ]:
#fileList = glob.glob(folder_path + '*/*0001*/*/*obl*' , recursive=True)
fileList = glob.glob(folder_path + '*/*/*/*obl*' , recursive=True)
dcm_metadata = []

print(len(fileList))
counter = 0

# Remove all JPG
remove_jpg(output_folder_path)

# Sort List
fileList.sort()

for folder in tqdm(fileList,desc = 'Progress Bar: Processing'  ):
            
    # Convert dicom to JPEG
    jpg_list, dcm_values, output_folder = process_folder(folder+"/")
    
    # Append Metadata to FinalList
    dcm_metadata.append(dcm_values)
    
    # Crop Image
    crop_image(output_folder+"/",x_crop,y_crop)
    
    # Make collage
    make_patient_image(output_folder+"/",images_to_use)
    
    counter = counter + 1

In [ ]:
# Create Empty DataFrame
df = pd.DataFrame(columns=['name','age','size','weight','ethnic_grp','occupation','smoking_status'])

# Get Patient List
counter = 0
for entry in dcm_metadata:
    df.loc[counter] = [entry[0][0]] + [entry[0][1]] + [entry[0][2]] + [entry[0][3]] + [entry[0][4]] + [entry[0][5]] + [entry[0][6]]
    counter = counter + 1

df.to_csv("patients.csv", sep=',', encoding='utf-8', index=False)

In [ ]:
print(df.head())

In [ ]:
#Target Flags
labels_file = "/kaggle/input/prostate-mri-us-biopsy/TCIA Biopsy Data_2020-07-14.xlsx"
file_name = "final.jpg"
folder_path = "/kaggle/working/kaggle/input/prostate-mri-us-biopsy/prostate_mri_us_biopsy/Prostate-MRI-US-Biopsy"

labels_df = pd.read_excel (labels_file)
fileList = glob.glob(folder_path + '**/*/*/*/'+file_name   , recursive=True)
#print(fileList[0:2])

for image in fileList:
    #print the patient ID's
    print(image[121:125])

In [ ]:
#display a few final jpg images
# Limit to 10 images
max_images = 10
selected_images = fileList[:max_images]

# Plot the images
fig, axes = plt.subplots(1, len(selected_images), figsize=(15, 5))

for i, image_path in enumerate(selected_images):
    img = cv2.imread(image_path)  # Read image
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # Convert to RGB for displaying
    
    # Display the image
    axes[i].imshow(img)
    axes[i].axis('off')  # Hide axes
    axes[i].set_title(f"Patient: {image_path[121:125]}")  # Display Patient ID

plt.tight_layout()
plt.show()

In [ ]:
#we can now convert the jpg data into a pixel df with the label using percent cancer in core
def getLabel(patient_id):
    #here check the XL sheet for the patient percent cancer if greater than 0 return 1 else 0
    patient_df =  labels_df[labels_df['Patient Number'].str.contains(patient_id)]
    for idx,row in patient_df.iterrows():
        if(row['% Cancer in Core'] > 0):
            return 1
    return 0

In [ ]:
batch_size = 300  # Adjust based on memory
new_size = (128, 128)

# Function to process a single batch
def process_batch(file_list, start_idx, batch_size):
    end_idx = min(start_idx + batch_size, len(file_list))
    batch_pixel_data = []
    batch_labels = []
    
    for image_path in file_list[start_idx:end_idx]:
        # Ensure the file exists
        if not os.path.isfile(image_path):
            print(f"File not found: {image_path}")
            continue
        
        # Read image
        im = cv2.imread(image_path)
        if im is None:
            print(f"Error reading image: {image_path}")
            continue
        
        # Resize and flatten image
        im_resized = cv2.resize(im, new_size)
        pixel_data = im_resized.flatten()
        
        # Extract label
        label_segment = image_path[121:125]  # Adjust indices as needed
        label = getLabel(label_segment)
        
        # Append to batch data
        batch_pixel_data.append(pixel_data)
        batch_labels.append(label)
    
    return batch_pixel_data, batch_labels

# Main processing loop
def process_images_in_batches(file_list, batch_size, output_file="/kaggle/working/processed_images_with_labels.csv"):
    # Initialize storage for the DataFrame
    columns = [f'pixel_{i}' for i in range(new_size[0] * new_size[1] * 3)] + ['output_label']
    
    # Write the header for the output file
    pd.DataFrame(columns=columns).to_csv(output_file, index=False)
    
    # Process in batches
    for start_idx in range(0, len(file_list), batch_size):
        # Process current batch
        batch_pixel_data, batch_labels = process_batch(file_list, start_idx, batch_size)
        
        # Create a DataFrame for the batch
        batch_df = pd.DataFrame(batch_pixel_data, columns=columns[:-1])
        batch_df['output_label'] = batch_labels
        
        # Append to CSV
        batch_df.to_csv(output_file, mode='a', header=False, index=False)
        
        # Print progress
        print(f"Processed batch {start_idx // batch_size + 1} / {len(file_list) // batch_size + 1}")
    
    print("Processing complete.")

# Call the function
process_images_in_batches(fileList, batch_size)

In [ ]:
path_to_df = "/kaggle/working/processed_images_with_labels.csv"
df_final = pd.read_csv(path_to_df)

In [ ]:
print(df_final.head)

In [ ]:
#scale the pixels
first_image_pixels = df_final.iloc[0, :-1].values  # Exclude the label column

# Reshape to the original dimensions (128, 128, 3)
first_image = first_image_pixels.reshape(128, 128, 3).astype(np.uint8)

# Display the image
plt.imshow(first_image)
plt.axis('off')  # Hide axes
plt.title("First Image from DataFrame")
plt.show()

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Separate features and labels
features = df_final.iloc[:, :-1]  # All columns except 'output_label'
labels = df_final['output_label']  # 'output_label' column

# Initialize the scaler (Min-Max Scaling to [0, 1])
scaler = MinMaxScaler()

# Scale the features
scaled_features = scaler.fit_transform(features)

# Combine scaled features and labels back into a DataFrame
scaled_df_final = pd.DataFrame(scaled_features, columns=features.columns)
scaled_df_final['output_label'] = labels.reset_index(drop=True)

# Display the first few rows of the scaled DataFrame
print(scaled_df_final.head())

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from skimage.feature import local_binary_pattern

# Feature extraction functions
def extract_mri_collage_features(image):
    # Convert to grayscale
    gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    # 1. Histogram of Intensities (Grayscale)
    hist = cv2.calcHist([gray_image], [0], None, [16], [0, 256]).flatten()  # 16 bins
    
    # 2. Edge Features (using Canny edge detector)
    edges = cv2.Canny(gray_image, 50, 150)
    edge_density = np.sum(edges) / (edges.shape[0] * edges.shape[1])  # Edge density
    
    # 3. Local Binary Pattern (LBP) for Texture
    lbp = local_binary_pattern(gray_image, P=8, R=1, method='uniform')
    lbp_hist = np.histogram(lbp.ravel(), bins=np.arange(0, 11), range=(0, 10))[0]  # LBP histogram
    
    # 4. Entropy as a measure of texture complexity
    hist_normalized = hist / hist.sum()
    entropy = -np.sum(hist_normalized * np.log2(hist_normalized + 1e-5))  # Entropy
    
    # Combine features
    features = np.hstack([hist, edge_density, lbp_hist, entropy])
    return features

# Function to process a single batch of MRI collages
def process_batch_with_mri_features(file_list, start_idx, batch_size, new_size):
    end_idx = min(start_idx + batch_size, len(file_list))
    batch_feature_data = []
    batch_labels = []
    
    for image_path in file_list[start_idx:end_idx]:
        # Ensure the file exists
        if not os.path.isfile(image_path):
            print(f"File not found: {image_path}")
            continue
        
        # Read image
        im = cv2.imread(image_path)
        if im is None:
            print(f"Error reading image: {image_path}")
            continue
        
        # Resize image
        im_resized = cv2.resize(im, new_size)
        
        # Extract features
        features = extract_mri_collage_features(im_resized)
        
        # Extract label (adjust based on your file naming convention)
        label_segment = image_path[121:125]  # Adjust indices as needed
        label = getLabel(label_segment)
        
        # Append to batch data
        batch_feature_data.append(features)
        batch_labels.append(label)
    
    return batch_feature_data, batch_labels

# Main processing loop
def process_mri_collages_with_features(file_list, batch_size, output_file="/kaggle/working/mri_collage_features.csv", new_size=(128, 128)):
    # Initialize columns for the features
    feature_names = [f'gray_hist_{i}' for i in range(16)] + ['edge_density'] + [f'lbp_hist_{i}' for i in range(10)] + ['entropy']
    columns = feature_names + ['output_label']
    
    # Write the header for the output file
    pd.DataFrame(columns=columns).to_csv(output_file, index=False)
    
    # Process in batches
    for start_idx in range(0, len(file_list), batch_size):
        # Process current batch
        batch_feature_data, batch_labels = process_batch_with_mri_features(file_list, start_idx, batch_size, new_size)
        
        # Create a DataFrame for the batch
        batch_df = pd.DataFrame(batch_feature_data, columns=feature_names)
        batch_df['output_label'] = batch_labels
        
        # Append to CSV
        batch_df.to_csv(output_file, mode='a', header=False, index=False)
        
        # Print progress
        print(f"Processed batch {start_idx // batch_size + 1} / {len(file_list) // batch_size + 1}")
    
    print("Processing complete.")

# Call the function
process_mri_collages_with_features(fileList, batch_size)

In [ ]:
path_to_df_f = "/kaggle/working/mri_collage_features.csv"
df_final_features = pd.read_csv(path_to_df_f)

In [ ]:
print(df_final_features.head)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Separate features and labels
features = df_final_features.iloc[:, :-1]  # All columns except 'output_label'
labels = df_final_features['output_label']  # 'output_label' column

# Initialize the scaler (Min-Max Scaling to [0, 1])
scaler = MinMaxScaler()

# Scale the features
scaled_features = scaler.fit_transform(features)

# Combine scaled features and labels back into a DataFrame
scaled_df_final_features = pd.DataFrame(scaled_features, columns=features.columns)
scaled_df_final_features['output_label'] = labels.reset_index(drop=True)

# Display the first few rows of the scaled DataFrame
print(scaled_df_final_features.head())

In [ ]:
from sklearn.model_selection import train_test_split

# Assuming `scaled_df_final` is your DataFrame

# Separate features and labels
X = scaled_df_final.iloc[:, :-1]  # All columns except 'output_label'
y = scaled_df_final['output_label']  # 'output_label' column

# Perform train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Print the shapes of the resulting splits
print("Training features shape:", X_train.shape)
print("Testing features shape:", X_test.shape)
print("Training labels shape:", y_train.shape)
print("Testing labels shape:", y_test.shape)

In [ ]:
from sklearn.model_selection import train_test_split

# Assuming `scaled_df_final` is your DataFrame

# Separate features and labels
X_features = scaled_df_final_features.iloc[:, :-1]  # All columns except 'output_label'
y_features = scaled_df_final_features['output_label']  # 'output_label' column

# Perform train-test split
X_train_features, X_test_features, y_train_features, y_test_features = train_test_split(X_features, y_features, test_size=0.2, random_state=42)

# Print the shapes of the resulting splits
print("Training features shape:", X_train_features.shape)
print("Testing features shape:", X_test_features.shape)
print("Training labels shape:", y_train_features.shape)
print("Testing labels shape:", y_test_features.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

# Initialize the Logistic Regression model
log_reg = LogisticRegression(max_iter=1000)  # You can adjust the max_iter if needed

# Fit the model to the training data
log_reg.fit(X_train, y_train)

# Make predictions on the test set
y_pred = log_reg.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

# Print the results
print("Accuracy:", accuracy)
print("Confusion Matrix:\n", conf_matrix)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

# Initialize the Logistic Regression model
log_reg_f = LogisticRegression(max_iter=1000)  # You can adjust the max_iter if needed

# Fit the model to the training data
log_reg_f.fit(X_train_features, y_train_features)

# Make predictions on the test set
y_pred_f = log_reg_f.predict(X_test_features)

# Evaluate the model
accuracy_f = accuracy_score(y_test_features, y_pred_f)
conf_matrix_f = confusion_matrix(y_test_features, y_pred_f)

# Print the results
print("Accuracy:", accuracy_f)
print("Confusion Matrix:\n", conf_matrix_f)

In [ ]:
# Make predictions on the test set
y_pred_log_train = log_reg.predict(X_train)
# Evaluate the model
accuracy_log_train = accuracy_score(y_train, y_pred_log_train)
print("Accuracy:",accuracy_log_train)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# Fit the model
rf_model.fit(X_train, y_train)

# Predict on test set
y_pred_rf = rf_model.predict(X_test)

# Evaluate the model
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print("Accuracy:",accuracy_rf)
print("Random Forest Classifier Performance:")
print(classification_report(y_test, y_pred_rf))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

rf_model_f = RandomForestClassifier(n_estimators=100, random_state=42)

# Fit the model
rf_model_f.fit(X_train_features, y_train_features)

# Predict on test set
y_pred_rf_f = rf_model_f.predict(X_test_features)

# Evaluate the model
accuracy_rf_f = accuracy_score(y_test_features, y_pred_rf_f)
print("Accuracy:",accuracy_rf_f)
print("Random Forest Classifier Performance:")
print(classification_report(y_test_features, y_pred_rf_f))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# Fit the model
rf_model.fit(X_train, y_train)

# Predict on test set
y_pred_rf = rf_model.predict(X_test)

# Evaluate the model
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print("Accuracy:",accuracy_rf)
print("Random Forest Classifier Performance:")
print(classification_report(y_test, y_pred_rf))

In [ ]:
# Make predictions on the test set
y_pred_rf_train = rf_model.predict(X_train)
# Evaluate the model
accuracy_rf_train = accuracy_score(y_train, y_pred_rf_train)
print("Accuracy:",accuracy_rf_train)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Initialize the Decision Tree classifier
dt_clf = DecisionTreeClassifier(random_state=42)

# Train the model
dt_clf.fit(X_train, y_train)

# Make predictions on the test set
y_pred_dt = dt_clf.predict(X_test)

# Evaluate the model
accuracy_dt = accuracy_score(y_test, y_pred_dt)
print("Accuracy:",accuracy_dt)

# Evaluate the model
print("Decision Tree Classification Report:")
print(classification_report(y_test, y_pred_dt))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_dt))

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Initialize the Decision Tree classifier
dt_clf_f = DecisionTreeClassifier(random_state=42)

# Train the model
dt_clf_f.fit(X_train_features, y_train_features)

# Make predictions on the test set
y_pred_dt_f = dt_clf_f.predict(X_test_features)

# Evaluate the model
accuracy_dt_f = accuracy_score(y_test_features, y_pred_dt_f)
print("Accuracy:",accuracy_dt_f)

# Evaluate the model
print("Decision Tree Classification Report:")
print(classification_report(y_test_features, y_pred_dt_f))

print("Confusion Matrix:")
print(confusion_matrix(y_test_features, y_pred_dt_f))

In [ ]:
y_pred_dt_train = dt_clf.predict(X_train)
# Evaluate the model
accuracy_dt_train = accuracy_score(y_train, y_pred_dt_train)
print("Accuracy:",accuracy_dt_train)

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix

# Initialize the Gaussian Naive Bayes classifier
nb_clf = GaussianNB()

# Train the model
nb_clf.fit(X_train, y_train)

# Make predictions on the test set
y_pred_nb = nb_clf.predict(X_test)

# Evaluate the model
accuracy_nb = accuracy_score(y_test, y_pred_nb)
print("Accuracy:",accuracy_nb)

print("Naive Bayes Classification Report:")
print(classification_report(y_test, y_pred_nb))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_nb))

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix

# Initialize the Gaussian Naive Bayes classifier
nb_clf_f = GaussianNB()

# Train the model
nb_clf_f.fit(X_train_features, y_train_features)

# Make predictions on the test set
y_pred_nb_f = nb_clf_f.predict(X_test_features)

# Evaluate the model
accuracy_nb_f = accuracy_score(y_test_features, y_pred_nb_f)
print("Accuracy:",accuracy_nb_f)

print("Naive Bayes Classification Report:")
print(classification_report(y_test_features, y_pred_nb_f))

print("Confusion Matrix:")
print(confusion_matrix(y_test_features, y_pred_nb_f))

In [ ]:
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import classification_report, confusion_matrix

# Initialize the Bernoulli Naive Bayes classifier
nb_clf = BernoulliNB()

# Train the model
nb_clf.fit(X_train, y_train)

# Make predictions on the test set
y_pred_nb = nb_clf.predict(X_test)

# Evaluate the model
accuracy_nb = accuracy_score(y_test, y_pred_nb)
print("Accuracy:",accuracy_nb)

# Evaluate the model
print("Naive Bayes Classification Report:")
print(classification_report(y_test, y_pred_nb))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_nb))

In [ ]:
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import classification_report, confusion_matrix

# Initialize the Bernoulli Naive Bayes classifier
nb_clf_f = BernoulliNB()

# Train the model
nb_clf_f.fit(X_train_features, y_train_features)

# Make predictions on the test set
y_pred_nb_f = nb_clf_f.predict(X_test_features)

# Evaluate the model
accuracy_nb_f = accuracy_score(y_test_features, y_pred_nb_f)
print("Accuracy:",accuracy_nb_f)

# Evaluate the model
print("Naive Bayes Classification Report:")
print(classification_report(y_test_features, y_pred_nb_f))

print("Confusion Matrix:")
print(confusion_matrix(y_test_features, y_pred_nb_f))

In [ ]:
# Make predictions on the test set
y_pred_nb_train = nb_clf.predict(X_train)
# Evaluate the model
accuracy_nb_train = accuracy_score(y_train, y_pred_nb_train)
print("Accuracy:",accuracy_nb_train)

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

# Example: Assume you already have X_train, X_test, y_train, y_test

# List of activation functions to test
activation_functions = ['relu', 'tanh', 'logistic']

# Store results in a list
results = []

# Loop through each activation function
for activation in activation_functions:
    print(f"\nTesting MLP with {activation} activation function...")

    # Initialize the MLPClassifier with the current activation function
    mlp_clf = MLPClassifier(hidden_layer_sizes=(128, 64, 32),  # Example architecture
                            activation=activation,          # Current activation function
                            solver='adam',                  # Optimizer
                            max_iter=200,                   # Max iterations
                            random_state=42,                # Reproducibility
                            batch_size=128,                 # Batch size
                            learning_rate_init=0.001)       # Learning rate

    # Train the model
    mlp_clf.fit(X_train, y_train)

    # Make predictions on the test set
    y_pred = mlp_clf.predict(X_test)

    # Calculate accuracy
    accuracy = accuracy_score(y_test, y_pred)

    # Get classification report
    class_report = classification_report(y_test, y_pred)

    # Append results to the list
    results.append({
        'Activation Function': activation,
        'Accuracy': accuracy,
        'Classification Report': class_report
    })

    print(f"Accuracy for {activation}: {accuracy:.4f}")
    print(f"Classification Report for {activation}:\n{class_report}")

# Convert results into a DataFrame for better readability
results_df = pd.DataFrame(results)

# Print the final results (Accuracy and Classification Reports)
print("\nFinal Results (Activation Function vs Accuracy and Classification Report):")
print(results_df)

In [ ]:
# from sklearn.model_selection import GridSearchCV
# from sklearn.neural_network import MLPClassifier

# # Define your model
# model = MLPClassifier(random_state=42)

# # Define the parameter grid to search over
# param_grid = {
#     'hidden_layer_sizes': [(128, 64, 32)], 
#     'activation': ['tanh'],        
#     'solver': ['adam', 'sgd'],                                     # Optimizer options
#     'max_iter': [100, 200 ,300],                                         # Maximum iterations
#     'learning_rate_init': [0.001, 0.01 , 2e-5 , 2e-3]                            # Learning rate
# }

# # Initialize GridSearchCV with the model and parameter grid
# grid_search = GridSearchCV(estimator=model, 
#                            param_grid=param_grid, 
#                            cv=5,                # Cross-validation folds
#                            n_jobs=-1,           # Use all available CPU cores
#                            scoring='accuracy',  # Scoring metric (accuracy in this case)
#                            verbose=2)           # Verbosity level to see progress

# # Fit the grid search
# grid_search.fit(X_train, y_train)

# # Get the best parameters and best score
# print(f"Best Parameters: {grid_search.best_params_}")
# print(f"Best Accuracy: {grid_search.best_score_}")

# # Get the best model
# best_model = grid_search.best_estimator_

# # Make predictions with the best model
# y_pred = best_model.predict(X_test)

# # Evaluate the performance on test set
# from sklearn.metrics import accuracy_score, classification_report
# print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
# print(f"Classification Report:\n{classification_report(y_test, y_pred)}")

In [ ]:
print(combined_df)

In [ ]:
# Define the aggregation functions
aggregation_functions = {
    'PSA (ng/mL)': 'mean',  # Average PSA value per patient
    'Primary Gleason': 'max',  # Highest Primary Gleason score
    'Secondary Gleason': 'max',  # Highest Secondary Gleason score
    'Cancer Length (mm)': 'sum',  # Total cancer length
    '% Cancer in Core': 'mean',  # Average % cancer in core
    'Core Fragment #1 Tissue Length (mm)': 'mean',
    'Core Fragment #2 Tissue Length (mm)': 'mean',
    'Core Fragment #3 Tissue Length (mm)': 'mean',
    'Bx Tip X (MRI Coord)': 'mean',
    'Bx Tip Y (MRI Coord)': 'mean',
    'Bx Tip Z (MRI Coord)': 'mean',
    'Bx Base X (MRI Coord)': 'mean',
    'Bx Base Y (MRI Coord)': 'mean',
    'Bx Base Z (MRI Coord)': 'mean',
    'Bx Tip X (US Coord)': 'mean',
    'Bx Tip Y (US Coord)': 'mean',
    'Bx Tip Z (US Coord)': 'mean',
    'Bx Base X (US Coord)': 'mean',
    'Bx Base Y (US Coord)': 'mean',
    'Bx Base Z (US Coord)': 'mean',
    'Prostate Volume (CC)': 'mean',  # Average prostate volume
    'Core Label': lambda x: ', '.join(sorted(set(x.dropna()))),  # Unique, sorted core labels
    'UCLA Score (Similar to PIRADS v2)': 'max',  # Highest UCLA Score
    'ROI Volume (cc)': 'mean',  # Average ROI volume
    'Target No.': 'max',  # Highest Target Number
    'Patient ID': 'first'  # Retain first Patient ID (indexing clarity)
}

# Apply grouping and aggregation
combined_df_final = (
    combined_df
    .groupby('Patient ID', as_index=False)  # Group by 'Patient ID'
    .agg(aggregation_functions)  # Apply aggregation
)

# Display the final DataFrame
print(combined_df_final)

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

# Split 'Core Label' into lists of labels
core_label_lists = combined_df_final['Core Label'].str.split(', ')

# Apply MultiLabelBinarizer
mlb = MultiLabelBinarizer()
core_label_encoded = pd.DataFrame(mlb.fit_transform(core_label_lists), columns=mlb.classes_)

# Concatenate the binarized labels with the original DataFrame
combined_df_final = pd.concat([combined_df_final.reset_index(drop=True), core_label_encoded], axis=1)

# Drop the original 'Core Label' column
combined_df_final.drop(columns=['Core Label'], inplace=True)
print("MultiLabelBinarized 'Core Label' and removed the original column.")

In [ ]:
print(combined_df_final)

In [ ]:
# Drop the 'Patient ID' column
combined_df_final.drop(columns=['Patient ID'], inplace=True)
print("Dropped 'Patient ID' column.")

In [ ]:
print(combined_df_final.columns)

In [ ]:
# Impute missing values with 0
combined_df_final.fillna(0, inplace=True)
print("Replaced all NaN values with 0.")

In [ ]:
combined_df_final.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Separate features and target (assuming 'Target' is your target column)
X = combined_df_final.drop(columns=['UCLA Score (Similar to PIRADS v2)'])  # Adjust target column name if different
y = combined_df_final['UCLA Score (Similar to PIRADS v2)']

# Scale the feature data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled the feature data.")

In [ ]:
# Perform the train-test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print(f"Train and test data shapes:")
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}, y_test: {y_test.shape}")

In [ ]:
#start training models
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Train Logistic Regression model (multiclass)
log_reg = LogisticRegression(max_iter=1000, multi_class='ovr', random_state=42)
log_reg.fit(X_train, y_train)

# Predictions and evaluation
y_pred_log_reg = log_reg.predict(X_test)
accuracy_log_reg = accuracy_score(y_test, y_pred_log_reg)

print("\nLogistic Regression:")
print(f"Accuracy: {accuracy_log_reg}")
print(f"Classification Report:\n{classification_report(y_test, y_pred_log_reg)}")

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# Train Decision Tree Classifier model (multiclass)
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

# Predictions and evaluation
y_pred_dt = dt.predict(X_test)
accuracy_dt = accuracy_score(y_test, y_pred_dt)

print("\nDecision Tree Classifier:")
print(f"Accuracy: {accuracy_dt}")
print(f"Classification Report:\n{classification_report(y_test, y_pred_dt)}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Train Random Forest Classifier model (multiclass)
rf = RandomForestClassifier(random_state=42, n_estimators=100)
rf.fit(X_train, y_train)

# Predictions and evaluation
y_pred_rf = rf.predict(X_test)
accuracy_rf = accuracy_score(y_test, y_pred_rf)

print("\nRandom Forest Classifier:")
print(f"Accuracy: {accuracy_rf}")
print(f"Classification Report:\n{classification_report(y_test, y_pred_rf)}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

# Define models (including additional classifiers)
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(),
    'Random Forest': RandomForestClassifier(n_estimators=100),
    'Support Vector Classifier': SVC(),
    'K-Nearest Neighbors': KNeighborsClassifier(),
    'Gradient Boosting Classifier': GradientBoostingClassifier(n_estimators=100),
    'Naive Bayes': GaussianNB()
}

# Function to train and evaluate models without PCA
def model_analysis(X_train, X_test, y_train, y_test):
    results = {}

    # Train and evaluate each model
    for model_name, model in models.items():
        # Train the model
        model.fit(X_train, y_train)
        
        # Make predictions
        y_pred = model.predict(X_test)
        
        # Evaluate accuracy
        accuracy = accuracy_score(y_test, y_pred)
        
        results[model_name] = accuracy
    
    return results

# Perform model analysis without PCA
results = model_analysis(X_train, X_test, y_train, y_test)

# Print results
for model_name, accuracy in results.items():
    print(f"{model_name}: {accuracy:.4f}")

In [ ]:
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

# Define models (including additional classifiers)
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(),
    'Random Forest': RandomForestClassifier(n_estimators=100),
    'Support Vector Classifier': SVC(),
    'K-Nearest Neighbors': KNeighborsClassifier(),
    'Gradient Boosting Classifier': GradientBoostingClassifier(n_estimators=100),
    'Naive Bayes': GaussianNB()
}

# Function to perform PCA and evaluate the model
def pca_analysis(n_components_list, X_train, X_test, y_train, y_test):
    results = {}
    
    for n_components in n_components_list:
        # Apply PCA
        pca = PCA(n_components=n_components)
        X_train_pca = pca.fit_transform(X_train)
        X_test_pca = pca.transform(X_test)
        
        # Train and evaluate each model
        model_results = {}
        for model_name, model in models.items():
            model.fit(X_train_pca, y_train)
            y_pred = model.predict(X_test_pca)
            accuracy = accuracy_score(y_test, y_pred)
            model_results[model_name] = accuracy
        
        results[n_components] = model_results
    
    return results

# Define the number of components to test (e.g., 3, 4, 6, 7, 8, etc.)
n_components_list = [2, 3, 4, 6, 7, 8 , 9 , 10 , 11 , 12 , 13 , 14 , 15]

# Perform PCA analysis
results = pca_analysis(n_components_list, X_train, X_test, y_train, y_test)

# Print results
for n_components, model_results in results.items():
    print(f"Number of PCA components: {n_components}")
    for model_name, accuracy in model_results.items():
        print(f"  {model_name}: {accuracy:.4f}")
    print("-" * 40)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score , mean_absolute_error

# Models to train
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree Regression": DecisionTreeRegressor(random_state=42),
    "Random Forest Regression": RandomForestRegressor(n_estimators=100, random_state=42),
    "Support Vector Regression": SVR(),
    "K-Nearest Neighbors Regression": KNeighborsRegressor(),
    "Gradient Boosting Regression": GradientBoostingRegressor(n_estimators=100, random_state=42)
}

# Iterate through PCA components (2 to 9)
for n_components in range(2, 16):
    print(f"Training with {n_components} PCA components...")
    
    # Apply PCA for dimensionality reduction
    pca = PCA(n_components=n_components)
    X_train_pca = pca.fit_transform(X_train)
    X_test_pca = pca.transform(X_test)

    # Evaluate each model
    for model_name, model in models.items():
        # Train the model
        model.fit(X_train_pca, y_train)

        # Make predictions
        y_pred = model.predict(X_test_pca)

        # Evaluate performance
        mse = mean_squared_error(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        # Print the results
        print(f"{model_name} - PCA {n_components} components:")
        print(f"  MSE: {mse:.4f}, MAE: {mae:.4f}, R2: {r2:.4f}")
    print("-" * 50)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Models to train (including more models)
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree Regression": DecisionTreeRegressor(random_state=42),
    "Random Forest Regression": RandomForestRegressor(n_estimators=100, random_state=42),
    "Support Vector Regression": SVR(),
    "K-Nearest Neighbors Regression": KNeighborsRegressor(),
    "Gradient Boosting Regression": GradientBoostingRegressor(n_estimators=100, random_state=42),
}

# Iterate through models
for model_name, model in models.items():
    print(f"Training {model_name} without PCA...")

    # Train the model
    model.fit(X_train, y_train)

    # Make predictions
    y_pred = model.predict(X_test)

    # Evaluate performance
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    # Print the results
    print(f"{model_name}:")
    print(f"  MSE: {mse:.4f}, MAE: {mae:.4f}, R2: {r2:.4f}")
    print("-" * 50)

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Define regressor models
regressors = {
    "Linear Regression": LinearRegression(),
    "Decision Tree Regression": DecisionTreeRegressor(random_state=42),
    "Random Forest Regression": RandomForestRegressor(n_estimators=100, random_state=42),
    "Support Vector Regression": SVR(),
    "K-Nearest Neighbors Regression": KNeighborsRegressor(),
    "Gradient Boosting Regression": GradientBoostingRegressor(n_estimators=100, random_state=42)
}

# Define hyperparameter grids for each regressor
param_grids_regressors = {
    "Linear Regression": {},
    "Decision Tree Regression": {
        "max_depth": [None, 10, 20],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 2]
    },
    "Random Forest Regression": {
        "n_estimators": [50, 100, 200],
        "max_depth": [None, 10, 20],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 2]
    },
    "Support Vector Regression": {
        "kernel": ['linear', 'rbf'],
        "C": [0.1, 1, 10],
        "gamma": ['scale', 'auto']
    },
    "K-Nearest Neighbors Regression": {
        "n_neighbors": [3, 5, 7],
        "weights": ['uniform', 'distance']
    },
    "Gradient Boosting Regression": {
        "learning_rate": [0.01, 0.1, 0.5],
        "n_estimators": [50, 100],
        "max_depth": [3, 5]
    }
}

# Perform GridSearchCV for each regressor
for model_name, model in regressors.items():
    print(f"Grid search for {model_name}...")
    
    grid_search = GridSearchCV(model, param_grids_regressors[model_name], cv=5, n_jobs=-1, scoring='neg_mean_squared_error')
    grid_search.fit(X_train, y_train)
    
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)
    
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    print(f"Best parameters for {model_name}: {grid_search.best_params_}")
    print(f"Test MSE: {mse:.4f}, MAE: {mae:.4f}, R2: {r2:.4f}")
    print("-" * 40)

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

# Define classifier models
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(),
    'Random Forest': RandomForestClassifier(n_estimators=100),
    'Support Vector Classifier': SVC(),
    'K-Nearest Neighbors': KNeighborsClassifier(),
    'Gradient Boosting Classifier': GradientBoostingClassifier(n_estimators=100),
    'Naive Bayes': GaussianNB()
}

# Define hyperparameter grids for each classifier
param_grids_classifiers = {
    'Logistic Regression': {
        'C': [0.1, 1, 10],
        'solver': ['liblinear', 'saga']
    },
    'Decision Tree': {
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2]
    },
    'Random Forest': {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2]
    },
    'Support Vector Classifier': {
        'kernel': ['linear', 'rbf'],
        'C': [0.1, 1, 10],
        'gamma': ['scale', 'auto']
    },
    'K-Nearest Neighbors': {
        'n_neighbors': [3, 5, 7],
        'weights': ['uniform', 'distance']
    },
    'Gradient Boosting Classifier': {
        'learning_rate': [0.01, 0.1, 0.5],
        'n_estimators': [50, 100],
        'max_depth': [3, 5]
    },
    'Naive Bayes': {}
}

# Perform GridSearchCV for each classifier
for model_name, model in classifiers.items():
    print(f"Grid search for {model_name}...")
    
    grid_search = GridSearchCV(model, param_grids_classifiers[model_name], cv=5, n_jobs=-1, scoring='accuracy')
    grid_search.fit(X_train, y_train)
    
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    
    print(f"Best parameters for {model_name}: {grid_search.best_params_}")
    print(f"Test Accuracy: {accuracy:.4f}")
    print("-" * 40)